# 🔴 講師デモ用 ｜ grover_demo_2qubit.ipynb

**2 qubit Grover アルゴリズム ― オラクル差し替えで汎用性を見せる**

---

## このノートブックの役割

- **前半 (セル 1-10)**: 講義中に講師がライブ実行するデモ
  - 基本 Grover (解 = q0=1∧q1=1) を 1 回まわす
  - オラクルを 3 回差し替えて、解が変わることを見せる
  - 「Grover は判定できる条件なら何でも探せる」という山場
- **後半 (セル 11+)**: 学生持ち帰り用の拡張雛形
  - 3 qubit Grover (N=8) への拡張
  - XOR 判定ベースの oracle 設計
  - オラクル汎用性の実験

## Qiskit のビット順序 (重要、混乱しやすい)

測定結果の文字列 `'01'` は **右から読んで q0, q1** の順:
- `'01'` → q1=0, q0=**1**
- `'10'` → q1=1, q0=0
- `'11'` → q1=1, q0=1

このノートブックでは、**ket 記法 `|q0 q1⟩` (左から q0)** で明示します。
Qiskit 文字列とは **順序が逆** なので注意。

---


---
## 環境セットアップ (Google Colab の場合は毎回実行)

第1回ガイダンスの Setup ノートブックと同じ手順です。
ローカル環境 (VSCode 等) で既に Qiskit をインストール済みの場合はスキップしてOK。


In [ ]:
# Qiskit のインストール (1-2 分かかります)
!pip install -q qiskit[visualization] qiskit-aer

# バージョン確認
import qiskit
print(f"Qiskit version: {qiskit.__version__}")
print("インストール完了")

## セル 1 ｜ セットアップ


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

SHOTS = 1000
sim = AerSimulator()

print("セットアップ完了")


## セル 2 ｜ Grover 回路の骨格を関数化

2 qubit Grover (N=4, M=1) を次の部品に分けて組む:

1. **初期化**: q0, q1 に H → 4 要素の等重ね合わせ、q2 に X → H → |−⟩ アンシラ
2. **オラクル**: 条件を満たす基底状態だけに位相 −1 を立てる (位相キックバック)
3. **Diffusion**: 平均まわりの反転で「マーク済み振幅」を増幅
4. **測定**: q0, q1 を測る

N=4, M=1 なら 1 反復で確率 1。


In [ ]:
def grover_2qubit(build_oracle):
    """
    2 qubit Grover 回路を組む
    build_oracle: (qc) → None, oracle ゲートを qc に in-place で追加する関数
    """
    qc = QuantumCircuit(3, 2)

    # --- 1. 初期化 ---
    qc.h([0, 1])          # q0, q1 を重ね合わせ
    qc.x(2); qc.h(2)      # q2 を |−⟩ に準備 (位相キックバック用アンシラ)
    qc.barrier()

    # --- 2. オラクル ---
    build_oracle(qc)
    qc.barrier()

    # --- 3. Diffusion (平均まわりの反転) ---
    # (H⊗H)(X⊗X) CZ (X⊗X)(H⊗H) の形
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    qc.barrier()

    # --- 4. 測定 ---
    qc.measure([0, 1], [0, 1])
    return qc


def run_and_show(qc, title):
    """実行してヒストグラムを表示"""
    counts = sim.run(qc, shots=SHOTS).result().get_counts()
    print(f"【{title}】")
    print(f"  測定結果: {counts}")
    # Qiskit 順 → ket 記法に変換して表示
    for bitstr, c in counts.items():
        q1, q0 = bitstr[0], bitstr[1]
        print(f"  → |q0 q1⟩ = |{q0} {q1}⟩  ({c} 回)")
    print()
    return counts


print("grover_2qubit() と run_and_show() を定義しました")


## セル 3 ｜ Demo 1 ｜ 解 = |q0=1, q1=1⟩ (基本 Toffoli オラクル)

**オラクルが判定する条件**: $q_0 \land q_1 = 1$

Toffoli ゲート (CCX) 単体でこの条件をそのまま書ける。q2 が |−⟩ に居るので、
条件を満たす基底状態 (q0=1, q1=1) だけに位相 −1 が立つ。

> **🎤 講師トーク**: 「Grover が |11⟩ を見つけました。
> 『|11⟩ をハードコードしたんだろう』と思いましたか? 半分正解、半分違います。
> このオラクルは Toffoli で、`q0 AND q1 = 1` という論理条件を判定しています。
> 結果 |11⟩ が出たのは、それが条件を満たす唯一の x だったから。」


In [ ]:
def oracle_11(qc):
    """解 = |q0=1, q1=1⟩ を探すオラクル: Toffoli そのもの"""
    qc.ccx(0, 1, 2)


qc1 = grover_2qubit(oracle_11)
counts1 = run_and_show(qc1, "Demo 1: オラクル = Toffoli,  解 = |q0=1, q1=1⟩")

# 回路図を描画 (確認用)
print("回路図:")
print(qc1.draw())


## セル 4 ｜ Demo 2 ｜ 解 = |q0=1, q1=0⟩ ─ オラクル差し替え

**オラクルが判定する条件**: $q_0 \land \lnot q_1 = 1$

Toffoli の前後に q1 に X をかける。これは「q1 の 0/1 を一時的に反転して Toffoli を通す」発想:

- **前の X**: q1 の意味を反転 (もし q1=0 なら、内部的に 1 として Toffoli に渡る)
- **Toffoli**: 「q0=1 AND q1'=1」を判定
- **後の X**: q1 の値を元に戻す

結果として「q0=1 AND q1=0」のとき位相 −1 が立つ。

> **🎤 講師トーク**: 「オラクルを差し替えただけで、別の解が浮かび上がりました。
> Grover 本体 (Diffusion、重ね合わせ、測定) は何も変わっていません。
> **変わったのは判定条件だけ**。」


In [ ]:
def oracle_10(qc):
    """解 = |q0=1, q1=0⟩ を探すオラクル"""
    qc.x(1)              # q1 を反転
    qc.ccx(0, 1, 2)      # Toffoli
    qc.x(1)              # q1 を戻す


qc2 = grover_2qubit(oracle_10)
counts2 = run_and_show(qc2, "Demo 2: X(q1) + Toffoli + X(q1),  解 = |q0=1, q1=0⟩")


## セル 5 ｜ Demo 3 ｜ 解 = |q0=0, q1=1⟩

**オラクルが判定する条件**: $\lnot q_0 \land q_1 = 1$

Toffoli の前後に q0 に X をかける。


In [ ]:
def oracle_01(qc):
    """解 = |q0=0, q1=1⟩ を探すオラクル"""
    qc.x(0)
    qc.ccx(0, 1, 2)
    qc.x(0)


qc3 = grover_2qubit(oracle_01)
counts3 = run_and_show(qc3, "Demo 3: X(q0) + Toffoli + X(q0),  解 = |q0=0, q1=1⟩")


## セル 6 ｜ Demo 4 ｜ 解 = |q0=0, q1=0⟩

**オラクルが判定する条件**: $\lnot q_0 \land \lnot q_1 = 1$

Toffoli の前後に q0, q1 両方に X をかける。


In [ ]:
def oracle_00(qc):
    """解 = |q0=0, q1=0⟩ を探すオラクル"""
    qc.x([0, 1])
    qc.ccx(0, 1, 2)
    qc.x([0, 1])


qc4 = grover_2qubit(oracle_00)
counts4 = run_and_show(qc4, "Demo 4: X(0,1) + Toffoli + X(0,1),  解 = |q0=0, q1=0⟩")


## セル 7 ｜ 4 つのデモをまとめて可視化

Grover 本体は何も変えず、オラクルだけ入れ替えた 4 実行の結果を並べます。


In [ ]:
# 横並びヒストグラム
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

demos = [
    (counts1, "Toffoli\n→ |q0=1, q1=1⟩"),
    (counts2, "X(q1) + Toffoli + X(q1)\n→ |q0=1, q1=0⟩"),
    (counts3, "X(q0) + Toffoli + X(q0)\n→ |q0=0, q1=1⟩"),
    (counts4, "X(0,1) + Toffoli + X(0,1)\n→ |q0=0, q1=0⟩"),
]

# 4 種の基底状態ラベル (Qiskit 測定順: q1 q0 右から左)
labels = ["00", "01", "10", "11"]

for ax, (counts, title) in zip(axes, demos):
    vals = [counts.get(lbl, 0) for lbl in labels]
    colors = ["#C00000" if v == max(vals) else "#C8C8C8" for v in vals]
    ax.bar(labels, vals, color=colors)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, SHOTS * 1.1)
    ax.set_xlabel("Qiskit 順 (q1 q0)")

plt.suptitle("同じ Grover 回路、オラクルだけ差し替えた 4 実行", fontsize=13)
plt.tight_layout()
plt.show()


## セル 8 ｜ デモのまとめ (学生に示す)

### 観察できたこと

| デモ | オラクル | 判定条件 | Grover の出す解 |
|:-:|:-|:-|:-:|
| 1 | Toffoli | $q_0 \land q_1$ | $\|q_0=1, q_1=1\rangle$ |
| 2 | X(q1) + Toffoli + X(q1) | $q_0 \land \lnot q_1$ | $\|q_0=1, q_1=0\rangle$ |
| 3 | X(q0) + Toffoli + X(q0) | $\lnot q_0 \land q_1$ | $\|q_0=0, q_1=1\rangle$ |
| 4 | X(両方) + Toffoli + X(両方) | $\lnot q_0 \land \lnot q_1$ | $\|q_0=0, q_1=0\rangle$ |

### 重要なポイント

- Grover 本体 (初期化、Diffusion、測定) は **全く同じ**
- **変わったのはオラクルだけ**
- オラクルは「判定条件」を量子回路で表現したもの

### この汎用性が何を意味するか

- Grover は **「判定できる条件なら何でも探せる」** 汎用ツール
- 来週 (#4) は **ライツアウトの「全消灯判定」** をオラクルに組み込み、パズルを Grover で解く
- 判定回路の部品は、今日学んだ **Toffoli、CNOT、X ゲート** で全部組める

### クエリ複雑度

- 古典全探索: $O(N)$ (N=4 なら最大 4 回試す)
- Grover: $O(\sqrt{N})$ (N=4 なら 1 回)
- N=16 なら 古典 16 vs Grover 4、 N=256 なら 古典 256 vs Grover 16

---


## セル 9 ｜ (補足) オラクルの中で実際に何が起きているか

Statevector で「オラクル直後」の状態を覗きます。
位相 −1 が「解に当たる基底状態」だけに立っていることを確認。


In [ ]:
# オラクル直後の状態を見る (Demo 1: 解 = |q0=1, q1=1⟩)

qc_peek = QuantumCircuit(3)
qc_peek.h([0, 1])
qc_peek.x(2); qc_peek.h(2)       # |−⟩ アンシラ
oracle_11(qc_peek)               # Toffoli

sv = Statevector(qc_peek)
print("オラクル直後の状態 (解=|11⟩):")
print(sv.draw('text'))
print()
print("→ |11⟩ の振幅だけに位相 −1 が乗っている (q2=|−⟩ の部分は共通)")
print("  Diffusion はこの位相情報を確率に変換する")


## セル 10 ｜ 🎤 講師の締めトーク + #4 予告

> **「今日の Grover デモで見せたのは 3 つ」**:
>
> 1. Grover は √N 反復で探索問題を解く (古典の N 倍速)
> 2. Grover の本体 (初期化 + Diffusion + 測定) は固定で、**オラクル設計が腕の見せどころ**
> 3. 判定条件を量子回路で表現できれば、何でも探索できる
>
> **「来週 (#4) は、皆さんが自分でオラクルを設計します」**:
>
> - 題材: タカラトミーの **ライツアウトパズル** 2×2 (4 マス、$2^4 = 16$ 通りの押し方)
> - オラクル: 「全消灯になる押し方」を判定する回路 (today の Toffoli と半加算器の発想で組む)
> - 中間レポート: 3×3 ライツアウトへの拡張 (期限: 2 週間後)

---

# 以下: 学生持ち帰り拡張用 (任意、挑戦歓迎)


## セル 11 ｜ 🎓 拡張 1 ｜ 3 qubit Grover (N=8)

探索空間を 8 要素に拡張します。

$N=8$ のとき、Grover の最適反復回数は $\lfloor \frac{\pi}{4} \sqrt{N} \rfloor = 2$ 回。

### ヒント

- 探索 qubit: q0, q1, q2 (3 つ)
- アンシラ: q3 を |−⟩ に
- オラクル: 解に当たる状態だけに位相 −1 を立てる (例: 解 = |111⟩ なら **3 制御 Toffoli (MCX)** 相当が必要)
- Diffusion: $(H^{\otimes 3})(X^{\otimes 3}) \cdot C^2Z \cdot (X^{\otimes 3})(H^{\otimes 3})$
  - $C^2Z$ (2 制御 Z) は Qiskit では `qc.h(2); qc.ccx(0,1,2); qc.h(2)` で代用可能
- 反復回数 2 (Grover ループを 2 回まわす)

### 雛形


In [ ]:
# 3 qubit Grover 雛形 (学生が完成させる)

def grover_3qubit(solution_oracle, n_iters=2):
    """
    3 qubit Grover (N=8)
    solution_oracle: (qc) → None, オラクルを in-place 追加する関数
    n_iters: Grover 反復回数 (N=8 なら 2 が最適)
    """
    qc = QuantumCircuit(4, 3)

    # 初期化
    qc.h([0, 1, 2])
    qc.x(3); qc.h(3)
    qc.barrier()

    for _ in range(n_iters):
        # オラクル
        solution_oracle(qc)
        qc.barrier()

        # Diffusion (TODO: 学生が実装)
        # ヒント:
        #   qc.h([0, 1, 2])
        #   qc.x([0, 1, 2])
        #   # C^2Z (|111⟩ の位相を反転): H + Toffoli + H で代用
        #   qc.h(2); qc.ccx(0, 1, 2); qc.h(2)
        #   qc.x([0, 1, 2])
        #   qc.h([0, 1, 2])

        # TODO: 上のヒントを参考に Diffusion を書く


        qc.barrier()

    qc.measure([0, 1, 2], [0, 1, 2])
    return qc


# 例: 解 = |q0=1, q1=1, q2=1⟩ のオラクル (3 入力 AND = Toffoli を MCX で代用)
def oracle_111_for_3qubit(qc):
    # 3 制御 NOT (q0, q1, q2 → q3) が必要だが 2 qubit Toffoli の組み合わせで作れる
    # 最も簡単: mcx を使う
    qc.mcx([0, 1, 2], 3)


# TODO: Diffusion を書き終えたら以下を実行
# qc_ext1 = grover_3qubit(oracle_111_for_3qubit, n_iters=2)
# counts_ext1 = sim.run(qc_ext1, shots=SHOTS).result().get_counts()
# print("解=|111⟩ (3 qubit, 2 反復):", counts_ext1)


## セル 12 ｜ 🎓 拡張 2 ｜ XOR 判定の Grover (⚠ 病的ケース教材)

AND ではなく **XOR** を判定条件にしてみる。

**判定条件**: $q_0 \oplus q_1 = 1$ (q0 と q1 が異なる)

解は 2 つある: $|q_0=0, q_1=1\rangle$ と $|q_0=1, q_1=0\rangle$

### ⚠ 注意: M = N/2 は Grover の病的ケース

この問題は **N=4, M=2** なので、$M/N = 1/2$。
**実は Grover では振幅を増幅できない** ─ これは Grover の有名な病的ケースです。

初期状態 (均等重ね合わせ) の時点で「解を測定する確率」は既に 50% ある。
Grover を 1 反復かけても、解の確率は 50% のまま (個別の 2 解にそれぞれ 25% ずつ)。
つまり **Grover を使わなくても、ランダムに測るだけで 50% の確率で解にぶつかる**。

### この拡張の目的

- XOR 判定の **オラクル構築技法** (補助 qubit + 位相キックバック + uncompute) を学ぶ
- その上で、Grover の **限界** を実際に手を動かして観察する
- 「どんな条件でも Grover が使える」わけではないことを知る

### XOR を量子回路で判定するには?

$q_0 \oplus q_1$ を Toffoli 制御に使いたい。$y \oplus x$ は CNOT で計算できるので:

1. 補助 qubit $q_{\text{tmp}}$ を用意 (|0⟩)
2. CNOT(q0, q_tmp), CNOT(q1, q_tmp) で $q_{\text{tmp}} = q_0 \oplus q_1$
3. $q_{\text{tmp}}$ を制御として、ターゲット (|−⟩ アンシラ) に CNOT → 位相キックバック
4. **逆操作** (CNOT を同じ順で 2 回) で $q_{\text{tmp}}$ を 0 に戻す (uncompute)

### 雛形


In [ ]:
# XOR 判定 Grover (学生が完成させる)

def grover_xor_2qubit():
    """
    解 = q0 ⊕ q1 = 1 を満たす状態を探す
    解は |01⟩, |10⟩ の 2 つ (M=2) ─ ただし M=N/2 は病的ケース
    """
    qc = QuantumCircuit(4, 2)   # q0, q1 = 探索, q2 = XOR 一時, q3 = |−⟩ アンシラ

    # 初期化
    qc.h([0, 1])
    qc.x(3); qc.h(3)
    qc.barrier()

    # --- XOR オラクル ---
    # TODO: 以下のヒントに従って実装
    # 1. q2 に q0 ⊕ q1 を計算: qc.cx(0, 2); qc.cx(1, 2)
    # 2. q2 を制御、q3 (|−⟩) をターゲットに CNOT → 位相キックバック
    # 3. q2 を 0 に戻す (uncompute): もう一度 qc.cx(1, 2); qc.cx(0, 2)


    qc.barrier()

    # Diffusion (2 qubit 版、上の grover_2qubit と同じ)
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    qc.barrier()

    qc.measure([0, 1], [0, 1])
    return qc


# TODO: オラクルを書いたら以下を実行
# qc_xor = grover_xor_2qubit()
# counts_xor = sim.run(qc_xor, shots=SHOTS).result().get_counts()
# print("XOR 判定 Grover (1 反復):", counts_xor)
# print()
# print("観察ポイント:")
# print("  - 解 (|01⟩, |10⟩) と 非解 (|00⟩, |11⟩) の出現頻度はどう違う?")
# print("  - 初期状態 (= Grover を1反復もかけない状態) で測定したら、どうなるはず?")
# print("  - Grover を 2 反復、3 反復かけたら、状況は改善するか? 悪化するか?")


## セル 13 ｜ 🎓 考察課題 (任意)

以下に自分の言葉で答える欄を置きます。

### 1. なぜ Grover 本体は変わらず、オラクルだけで「何を探すか」が決まるのか?

Grover の 3 部品 (初期化 / オラクル / Diffusion) それぞれの役割を説明してから答えてください。

**あなたの説明**: _______________________________________

---

### 2. XOR 判定 (セル 12) で Grover は解を増幅できない。なぜ?

ヒント: **M/N = 1/2** が鍵。Grover の成功確率の式 $\sin^2((2k+1)\theta)$ に $\sin\theta = \sqrt{M/N}$ を代入し、
$k = 0$ (Grover なし) と $k = 1$ (1 反復) で比較してみる。

さらに突っ込み:
- 反復回数 $k$ を増やすと、解の測定確率はどう変化する? (サイクルする? 単調に増える?)
- 「Grover は $M < N/2$ のとき有効」とよく言われるが、$M > N/2$ のときはどうなる?
  (実装して観察できる)

**あなたの説明**: _______________________________________

---

### 3. XOR 判定で uncompute (逆操作) が必要なのはなぜか?

ヒント: uncompute を省略すると q_tmp が 0 に戻らず、q0, q1 と **もつれたまま** Diffusion に入ってしまう。
その結果どうなるか、実際に uncompute なしの回路を書いて実験してみると分かる。

**あなたの説明**: _______________________________________

---

### 4. ライツアウト 2×2 のオラクルを設計するとしたら?

4 マスの盤面を q0, q1, q2, q3 で表し、q2 を押すと自分と隣接マスが反転する。
「全消灯」をオラクル判定するには、どんな回路を組めばよいか?

ヒント: 押し方を入力にして、半加算器の発想で「各マスが消えているか」を計算、
最後に「全マスが 0 か」を Toffoli で判定する。来週 #4 で扱う。

**あなたの考え**: _______________________________________

---

### 条件

- **AI 活用可**: ChatGPT, Claude など
- **最終的に自分の言葉で説明できる** ことが重要
- 実行結果 + 2-3 行のコメントを提出

---

> 🎯 このノートブックのメッセージ: 「Grover は汎用ツール。オラクル設計が設計者の腕」
